# 🛡️ GSL Modo 2 — Respuesta Adaptativa Integrada
**Geometric Signature Layer — PolicyAdapter + Ejecución por Nivel**

## Prerequisito
Este notebook consume los artefactos del **Modo 1** (`manifold_principal.json`,
`manifold_siem_*.json`, `dissonance_scores.csv`). Ejecutar Modo 1 al menos
un ciclo bimestral antes de activar Modo 2.

## Niveles de despliegue
```
Nivel 0 — Solo recomendación        (sin integración — un click ejecuta)
Nivel 1 — Webhook saliente          (endpoint HTTP del cliente)
Nivel 2 — API directa               (credenciales acotadas por acción)
Nivel 3 — Agente local              (SDK en infraestructura del cliente)
```
El cliente activa el nivel disponible por acción. Cada acción puede estar
en un nivel distinto — `snapshot_state` puede ser Nivel 2 mientras
`isolate_session` permanece en Nivel 0 hasta que el cliente confíe más.

## Lo que agrega sobre el Modo 1
- **PolicyAdapter**: dado score de disonancia + contexto → acción correcta
- **Motor de ejecución**: enruta la acción según nivel disponible
- **Override humano**: reversión en un click + señal de entrenamiento
- **Registro forense inmutable**: toda acción ejecutada queda trazada

## Lo que NO cambia
- La firma geométrica es la misma del Modo 1
- Los manifolds se cargan — no se reconstruyen
- El sistema sigue siendo paralelo a cualquier infraestructura del cliente

## Instalación de dependencias

In [ ]:
!pip install gradio numpy pandas matplotlib plotly requests -q
print('✅ Dependencias listas.')

## Bloque 1 — Carga del manifold y PolicyAdapter del Modo 1

Consume los puertos P3/P4 del ExportPackage del Modo 1.
Si no existe adapter entrenado, inicializa uno neutro con el diccionario
de acciones definido en `org_config.json`.

In [ ]:
import json, os, uuid, copy, hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

# ─── Config por defecto ───────────────────────────────────────
DEFAULT_CONFIG = {
    "org_name":     "Despacho Jurídico — Demo GSL",
    "org_type":     "legal",
    "principal_id": "ORG-DEMO-001",
    "bimester":     "2025-B1",
    "users": [
        {"id": "u01", "name": "Ana Socia (admin)",     "role": "admin",
         "typical_ips": ["192.168.1.10"],              "typical_hours": [8, 20]},
        {"id": "u02", "name": "Luis Colaborador Ext.", "role": "external",
         "typical_ips": ["10.0.0.5"],                 "typical_hours": [9, 18]},
        {"id": "u03", "name": "María Asociada",        "role": "staff",
         "typical_ips": ["192.168.1.12"],             "typical_hours": [9, 19]},
        {"id": "u04", "name": "Carlos Paralegal",      "role": "staff",
         "typical_ips": ["192.168.1.13"],             "typical_hours": [9, 18]}
    ],
    "assets": [
        {"id": "a01", "name": "Expediente Caso Alpha",  "sensitivity": "high"},
        {"id": "a02", "name": "Expediente Caso Beta",   "sensitivity": "high"},
        {"id": "a03", "name": "Contratos con Clientes", "sensitivity": "high"},
        {"id": "a04", "name": "Biblioteca Plantillas",  "sensitivity": "medium"},
        {"id": "a05", "name": "Agenda y Citas",         "sensitivity": "low"}
    ],
    "thresholds": {
        "geo_dissonance_alert":   0.25,
        "geo_dissonance_isolate": 0.55,
        "geo_dissonance_report":  0.40
    },
    # Diccionario de acciones con nivel requerido y reversibilidad
    "actions_dict": {
        "reduce_write_permissions": {
            "description": "Reduce permisos de escritura en el extremo",
            "reversible":  True,
            "impact":      "low",
            "min_dissonance": 0.25
        },
        "increase_validation_weight": {
            "description": "Añade latencia de validación a acciones del extremo",
            "reversible":  True,
            "impact":      "low",
            "min_dissonance": 0.25
        },
        "snapshot_state": {
            "description": "Crea copia inmutable del estado actual de activos",
            "reversible":  False,
            "impact":      "low",
            "min_dissonance": 0.25
        },
        "isolate_session": {
            "description": "Mueve sesión a sandbox de solo lectura",
            "reversible":  True,
            "impact":      "medium",
            "min_dissonance": 0.55
        },
        "terminate_session": {
            "description": "Cierra sesión con timeout genérico",
            "reversible":  False,
            "impact":      "high",
            "min_dissonance": 0.70
        },
        "revoke_credentials": {
            "description": "Suspende credenciales hasta verificación",
            "reversible":  True,
            "impact":      "high",
            "min_dissonance": 0.70
        },
        "freeze_affected_assets": {
            "description": "Marca activos accedidos para auditoría",
            "reversible":  True,
            "impact":      "medium",
            "min_dissonance": 0.55
        },
        "generate_incident_report": {
            "description": "Ensambla línea de tiempo forense completa",
            "reversible":  False,
            "impact":      "none",
            "min_dissonance": 0.25
        }
    },
    # Permisos por acción: nivel 0-3 por acción
    "action_levels": {
        "reduce_write_permissions":   0,
        "increase_validation_weight": 0,
        "snapshot_state":             0,
        "isolate_session":            0,
        "terminate_session":          0,
        "revoke_credentials":         0,
        "freeze_affected_assets":     0,
        "generate_incident_report":   0
    },
    # Endpoints por nivel (vacío = no configurado)
    "level1_webhook_url":  "",
    "level2_api_base":     "",
    "level2_api_key":      "",
    "level3_agent_socket": ""
}


def load_config(path: str = 'org_config.json') -> dict:
    p = Path(path)
    if p.exists():
        with open(p) as f:
            custom = json.load(f)
        merged = {**DEFAULT_CONFIG, **custom}
        for k in ('thresholds', 'actions_dict', 'action_levels'):
            if k in DEFAULT_CONFIG and k in custom:
                merged[k] = {**DEFAULT_CONFIG[k], **custom[k]}
        print(f'✅ Config cargada: {merged["org_name"]}')
    else:
        merged = copy.deepcopy(DEFAULT_CONFIG)
        print(f'ℹ️  Sin org_config.json — usando demo: {merged["org_name"]}')
    return merged


CFG      = load_config()
BIMESTER = CFG.get('bimester', '2025-B1')
RUN_ID   = f"GSL-M2-{CFG['principal_id']}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
ARTIFACTS = Path('gsl_m2_artifacts') / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f'  Run ID    : {RUN_ID}')
print(f'  Bimestre  : {BIMESTER}')
print(f'  Artefactos: {ARTIFACTS}')

In [ ]:
# ─── Carga de manifolds del Modo 1 ────────────────────────────

def find_latest_m1_artifacts(base: str = '.') -> Optional[Path]:
    """Busca el directorio de artefactos más reciente del Modo 1."""
    candidates = sorted(
        Path(base).glob('gsl_artifacts/GSL-M1-*'),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    return candidates[0] if candidates else None


def load_manifold(path: Path) -> dict:
    """Deserializa un manifold desde JSON — restaura np.arrays."""
    with open(path) as f:
        raw = json.load(f)
    return {k: {'vector': np.array(v['vector']),
                'n_events': v['n_events'],
                'entity_col': v.get('entity_col', 'user_id')}
            for k, v in raw.items()}


def load_m1_artifacts(artifact_dir: Optional[Path] = None) -> dict:
    """Carga todos los artefactos relevantes del Modo 1."""
    if artifact_dir is None:
        artifact_dir = find_latest_m1_artifacts()

    if artifact_dir is None or not artifact_dir.exists():
        print('⚠️  No se encontraron artefactos del Modo 1 — usando manifold sintético')
        return _synthetic_m1_artifacts()

    print(f'📂 Cargando artefactos del Modo 1: {artifact_dir}')
    result = {'artifact_dir': artifact_dir, 'manifolds': {}, 'scores': pd.DataFrame()}

    # Manifold principal
    mp = artifact_dir / 'manifold_principal.json'
    if mp.exists():
        result['manifolds']['principal'] = load_manifold(mp)
        print(f'  ✅ manifold_principal: {len(result["manifolds"]["principal"])} extremos')

    # Manifolds SIEM
    for f in artifact_dir.glob('manifold_siem_*.json'):
        node_id = f.stem.replace('manifold_siem_', '')
        result['manifolds'][f'siem_{node_id}'] = load_manifold(f)
        print(f'  ✅ manifold_siem_{node_id}: {len(result["manifolds"][f"siem_{node_id}"])} extremos')

    # Scores históricos
    sp = artifact_dir / 'dissonance_scores.csv'
    if sp.exists():
        result['scores'] = pd.read_csv(sp)
        print(f'  ✅ dissonance_scores: {len(result["scores"])} ventanas')

    return result


def _synthetic_m1_artifacts() -> dict:
    """Manifold sintético para demo cuando no hay Modo 1 previo."""
    import random
    rng = random.Random(42)
    manifold = {}
    for u in CFG['users']:
        manifold[u['id']] = {
            'vector':     np.array([rng.uniform(0.7,1.0) for _ in range(8)]),
            'n_events':   rng.randint(80, 200),
            'entity_col': 'user_id'
        }
    return {'artifact_dir': None,
            'manifolds': {'principal': manifold},
            'scores': pd.DataFrame()}


M1 = load_m1_artifacts()
print(f'\n✅ Manifolds disponibles: {list(M1["manifolds"].keys())}')

## Bloque 2 — PolicyAdapter y registro de permisos por nivel

El PolicyAdapter aprende del historial de decisiones del Modo 1.
Cada override humano actualiza los pesos con `W_CORRECTION = 2.0`.

El cliente declara qué nivel tiene disponible **por acción**.
Cada acción puede estar en un nivel distinto.

In [ ]:
# ─── PolicyAdapter ────────────────────────────────────────────

W_CONFIRM    = 1.0   # peso: decisión confirmada por humano
W_CORRECTION = 2.0   # peso: corrección humana (override)
W_LOW_CLARITY = 0.5  # peso: baja claridad en el contexto

ACTIONS_ORDERED = [
    'reduce_write_permissions',
    'increase_validation_weight',
    'snapshot_state',
    'isolate_session',
    'terminate_session',
    'revoke_credentials',
    'freeze_affected_assets',
    'generate_incident_report',
]

@dataclass
class PolicyAdapter:
    """
    Adapter ligero que mapea (dissonance, context) → action.
    Aprende del feedback humano bimestre a bimestre.
    """
    org_id:      str
    bimester:    str
    actions:     List[str] = field(default_factory=lambda: ACTIONS_ORDERED.copy())
    # Pesos por (acción, contexto_bucket) — contexto_bucket: 'low'|'med'|'high'|'critical'
    weights:     Dict[str, Dict[str, float]] = field(default_factory=dict)
    # Historial de decisiones para ajuste bimestral
    history:     List[dict] = field(default_factory=list)
    correction_rate: float = 0.0

    def __post_init__(self):
        if not self.weights:
            self._init_weights()

    def _init_weights(self):
        """Pesos iniciales basados en min_dissonance del diccionario."""
        buckets = ['low', 'med', 'high', 'critical']
        actions_cfg = CFG.get('actions_dict', {})
        for act in self.actions:
            min_d = actions_cfg.get(act, {}).get('min_dissonance', 0.25)
            # Peso inicial: más alto cuanto más coincide el umbral con el bucket
            self.weights[act] = {
                'low':      1.0 if min_d <= 0.25 else 0.1,
                'med':      1.0 if 0.25 < min_d <= 0.45 else 0.3,
                'high':     1.0 if 0.45 < min_d <= 0.65 else 0.2,
                'critical': 1.0 if min_d > 0.65 else 0.1,
            }

    def _dissonance_bucket(self, dis: float) -> str:
        if dis < 0.25:   return 'low'
        if dis < 0.45:   return 'med'
        if dis < 0.65:   return 'high'
        return 'critical'

    def recommend(self, dissonance: float, entity_id: str,
                  context: dict = None) -> List[Tuple[str, float]]:
        """
        Recomienda acciones ordenadas por peso para el nivel de disonancia dado.
        Devuelve lista de (acción, score) ordenada descendente.
        Solo incluye acciones cuyo min_dissonance es alcanzado.
        """
        bucket = self._dissonance_bucket(dissonance)
        actions_cfg = CFG.get('actions_dict', {})
        scored = []
        for act in self.actions:
            min_d = actions_cfg.get(act, {}).get('min_dissonance', 0.25)
            if dissonance < min_d:
                continue
            w = self.weights.get(act, {}).get(bucket, 0.1)
            scored.append((act, round(w, 4)))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def record_decision(self, action: str, dissonance: float,
                        confirmed: bool, overridden_to: str = None,
                        low_clarity: bool = False):
        """Registra una decisión humana y actualiza pesos."""
        bucket = self._dissonance_bucket(dissonance)
        weight = W_CONFIRM if confirmed else W_CORRECTION
        if low_clarity:
            weight *= W_LOW_CLARITY

        if confirmed:
            self.weights[action][bucket] = min(
                self.weights[action][bucket] * (1 + 0.1 * weight), 3.0)
        else:
            # Penalizar acción incorrecta
            self.weights[action][bucket] = max(
                self.weights[action][bucket] * (1 - 0.15 * weight), 0.01)
            # Reforzar la acción correcta
            if overridden_to and overridden_to in self.weights:
                self.weights[overridden_to][bucket] = min(
                    self.weights[overridden_to][bucket] * (1 + 0.2 * weight), 3.0)

        self.history.append({
            'timestamp':    datetime.now(timezone.utc).isoformat(),
            'action':       action,
            'dissonance':   dissonance,
            'bucket':       bucket,
            'confirmed':    confirmed,
            'overridden_to': overridden_to,
            'low_clarity':  low_clarity,
            'weight_applied': weight,
        })
        # Actualizar correction_rate
        if self.history:
            self.correction_rate = sum(
                1 for h in self.history if not h['confirmed']
            ) / len(self.history)

    def save(self, path: Path):
        state = {
            'org_id':   self.org_id,
            'bimester': self.bimester,
            'actions':  self.actions,
            'weights':  self.weights,
            'history':  self.history,
            'correction_rate': self.correction_rate,
            'saved_at': datetime.now(timezone.utc).isoformat(),
        }
        with open(path, 'w') as f:
            json.dump(state, f, indent=2)

    @classmethod
    def load(cls, path: Path) -> 'PolicyAdapter':
        with open(path) as f:
            state = json.load(f)
        pa = cls(org_id=state['org_id'], bimester=state['bimester'],
                 actions=state['actions'])
        pa.weights         = state['weights']
        pa.history         = state['history']
        pa.correction_rate = state.get('correction_rate', 0.0)
        return pa


# ─── Cargar o inicializar el adapter ─────────────────────────
ADAPTER_PATH = ARTIFACTS.parent.parent / f'policy_adapter_{CFG["principal_id"]}.json'

if ADAPTER_PATH.exists():
    adapter = PolicyAdapter.load(ADAPTER_PATH)
    print(f'✅ PolicyAdapter cargado: {ADAPTER_PATH}')
    print(f'   Bimestre anterior : {adapter.bimester}')
    print(f'   Decisiones en hist: {len(adapter.history)}')
    print(f'   Correction rate   : {adapter.correction_rate:.2%}')
    adapter.bimester = BIMESTER  # actualizar al bimestre actual
else:
    adapter = PolicyAdapter(
        org_id=CFG['principal_id'],
        bimester=BIMESTER,
        actions=list(CFG.get('actions_dict', {}).keys()) or ACTIONS_ORDERED
    )
    print(f'ℹ️  PolicyAdapter inicializado (sin historial previo)')
    print(f'   Acciones registradas: {len(adapter.actions)}')

print('\n✅ PolicyAdapter listo.')

In [ ]:
# ─── Registro de permisos por nivel ───────────────────────────

LEVEL_LABELS = {
    0: 'Nivel 0 — Recomendación (un click)',
    1: 'Nivel 1 — Webhook saliente',
    2: 'Nivel 2 — API directa',
    3: 'Nivel 3 — Agente local',
}

ACTION_LEVELS = CFG.get('action_levels', {k: 0 for k in ACTIONS_ORDERED})

print('📋 Permisos de ejecución por acción:')
print(f'  {"Acción":<35} {"Nivel":<45} {"Impacto":<10} {"Reversible"}')
print('  ' + '─'*100)
actions_cfg = CFG.get('actions_dict', {})
for act in adapter.actions:
    lvl  = ACTION_LEVELS.get(act, 0)
    cfg_a = actions_cfg.get(act, {})
    imp  = cfg_a.get('impact', 'unknown')
    rev  = '✓' if cfg_a.get('reversible', True) else '✗'
    lvl_label = LEVEL_LABELS.get(lvl, f'Nivel {lvl}')
    print(f'  {act:<35} {lvl_label:<45} {imp:<10} {rev}')

print('\nℹ️  Para cambiar niveles: editar "action_levels" en org_config.json')
print('   Ejemplo nivel 1: agregar "level1_webhook_url": "https://tu-servidor/gsl-webhook"')

## Bloque 3 — Motor de ejecución por nivel

Dado un evento de disonancia, el PolicyAdapter selecciona la acción
y este motor la enruta según el nivel configurado para esa acción.

Cada ejecución genera un **registro forense inmutable** con hash SHA-256.

In [ ]:
# ─── Registro forense ─────────────────────────────────────────

@dataclass
class ForensicRecord:
    record_id:       str
    timestamp:       str
    run_id:          str
    entity_id:       str
    action:          str
    level:           int
    dissonance:      float
    executed:        bool
    execution_result: str
    override:        bool = False
    override_by:     str  = ''
    override_action: str  = ''
    override_ts:     str  = ''
    override_token:  str  = ''
    sha256:          str  = ''

    def compute_hash(self) -> str:
        payload = json.dumps({
            'record_id': self.record_id,
            'timestamp': self.timestamp,
            'entity_id': self.entity_id,
            'action':    self.action,
            'dissonance': self.dissonance,
            'executed':  self.executed,
        }, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()[:16]

    def to_dict(self) -> dict:
        return {
            'record_id':        self.record_id,
            'timestamp':        self.timestamp,
            'run_id':           self.run_id,
            'entity_id':        self.entity_id,
            'action':           self.action,
            'level':            self.level,
            'level_label':      LEVEL_LABELS.get(self.level, f'Nivel {self.level}'),
            'dissonance':       self.dissonance,
            'executed':         self.executed,
            'execution_result': self.execution_result,
            'override':         self.override,
            'override_by':      self.override_by,
            'override_action':  self.override_action,
            'override_ts':      self.override_ts,
            'override_token':   self.override_token,
            'sha256':           self.sha256,
        }


FORENSIC_LOG: List[ForensicRecord] = []
ACTIVE_SESSIONS: Dict[str, dict]   = {}   # entity_id → estado actual


# ─── Ejecutores por nivel ─────────────────────────────────────

def execute_level0(action: str, entity_id: str,
                   dissonance: float, context: dict) -> Tuple[bool, str]:
    """
    Nivel 0 — presenta la recomendación. No ejecuta nada automáticamente.
    Devuelve (pendiente_confirmacion, mensaje).
    """
    msg = (f'[L0] Recomendación pendiente: {action} sobre {entity_id} '
           f'(disonancia={dissonance:.3f}). '
           f'Confirmar en el panel para ejecutar.')
    return False, msg   # executed=False hasta confirmación humana


def execute_level1(action: str, entity_id: str,
                   dissonance: float, context: dict,
                   webhook_url: str, override_token: str) -> Tuple[bool, str]:
    """
    Nivel 1 — envía payload al webhook del cliente.
    El cliente ejecuta la acción en su infraestructura.
    """
    import requests
    payload = {
        'event':          'gsl_action',
        'action':         action,
        'entity_id':      entity_id,
        'dissonance':     dissonance,
        'run_id':         RUN_ID,
        'reversible':     CFG.get('actions_dict',{}).get(action,{}).get('reversible', True),
        'override_token': override_token,
        'ttl_minutes':    30,
        'timestamp':      datetime.now(timezone.utc).isoformat(),
    }
    if not webhook_url:
        return False, '[L1] webhook_url no configurado — acción no enviada'
    try:
        r = requests.post(webhook_url, json=payload, timeout=10)
        r.raise_for_status()
        return True, f'[L1] Webhook enviado → HTTP {r.status_code}'
    except Exception as e:
        return False, f'[L1] Error webhook: {e}'


def execute_level2(action: str, entity_id: str,
                   dissonance: float, context: dict,
                   api_base: str, api_key: str) -> Tuple[bool, str]:
    """
    Nivel 2 — ejecuta directamente vía API del cliente.
    Las credenciales son acotadas: solo permisos para esta acción específica.
    """
    import requests
    if not api_base or not api_key:
        return False, '[L2] api_base o api_key no configurados'
    # Mapa de acción → endpoint REST estándar
    endpoint_map = {
        'reduce_write_permissions':   '/api/v1/permissions/restrict',
        'increase_validation_weight': '/api/v1/sessions/throttle',
        'snapshot_state':             '/api/v1/backup/snapshot',
        'isolate_session':            '/api/v1/sessions/sandbox',
        'terminate_session':          '/api/v1/sessions/terminate',
        'revoke_credentials':         '/api/v1/identity/suspend',
        'freeze_affected_assets':     '/api/v1/assets/freeze',
        'generate_incident_report':   '/api/v1/reports/incident',
    }
    endpoint = api_base.rstrip('/') + endpoint_map.get(action, f'/api/v1/actions/{action}')
    headers  = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    body     = {'entity_id': entity_id, 'dissonance': dissonance,
                'run_id': RUN_ID, 'timestamp': datetime.now(timezone.utc).isoformat()}
    try:
        r = requests.post(endpoint, json=body, headers=headers, timeout=10)
        r.raise_for_status()
        return True, f'[L2] API ejecutada → HTTP {r.status_code}'
    except Exception as e:
        return False, f'[L2] Error API: {e}'


def execute_level3(action: str, entity_id: str,
                   dissonance: float, context: dict,
                   agent_socket: str) -> Tuple[bool, str]:
    """
    Nivel 3 — instruye al agente local vía socket/IPC.
    El agente tiene los permisos reales — el GSL solo envía el nombre de la acción.
    """
    if not agent_socket:
        return False, '[L3] agent_socket no configurado'
    try:
        import socket
        msg = json.dumps({
            'action': action, 'entity_id': entity_id,
            'dissonance': dissonance, 'run_id': RUN_ID
        }).encode()
        with socket.socket(socket.AF_UNIX, socket.SOCK_STREAM) as s:
            s.connect(agent_socket)
            s.sendall(msg)
            resp = s.recv(1024).decode()
        return True, f'[L3] Agente respondió: {resp}'
    except Exception as e:
        return False, f'[L3] Error agente: {e}'


# ─── Dispatcher principal ─────────────────────────────────────

def dispatch_action(
    action:     str,
    entity_id:  str,
    dissonance: float,
    context:    dict = None,
    dry_run:    bool = False,
) -> ForensicRecord:
    """
    Enruta la acción al nivel configurado para ella.
    dry_run=True simula la ejecución sin enviar nada.
    """
    level         = ACTION_LEVELS.get(action, 0)
    override_token = str(uuid.uuid4())[:12]
    ts            = datetime.now(timezone.utc).isoformat()
    record_id     = f'REC-{str(uuid.uuid4())[:8].upper()}'

    if dry_run:
        executed, result = False, f'[DRY-RUN] Nivel {level} — {action} sobre {entity_id}'
    elif level == 0:
        executed, result = execute_level0(action, entity_id, dissonance, context or {})
    elif level == 1:
        executed, result = execute_level1(
            action, entity_id, dissonance, context or {},
            CFG.get('level1_webhook_url', ''), override_token)
    elif level == 2:
        executed, result = execute_level2(
            action, entity_id, dissonance, context or {},
            CFG.get('level2_api_base', ''), CFG.get('level2_api_key', ''))
    elif level == 3:
        executed, result = execute_level3(
            action, entity_id, dissonance, context or {},
            CFG.get('level3_agent_socket', ''))
    else:
        executed, result = False, f'Nivel {level} desconocido'

    rec = ForensicRecord(
        record_id=record_id, timestamp=ts, run_id=RUN_ID,
        entity_id=entity_id, action=action, level=level,
        dissonance=dissonance, executed=executed,
        execution_result=result, override_token=override_token
    )
    rec.sha256 = rec.compute_hash()
    FORENSIC_LOG.append(rec)

    # Actualizar estado de sesión activa
    if entity_id not in ACTIVE_SESSIONS:
        ACTIVE_SESSIONS[entity_id] = {
            'entity_id':    entity_id,
            'phase':        'normal',
            'restrictions': [],
            'dissonance':   dissonance,
            'last_action':  action,
            'last_ts':      ts,
        }
    sess = ACTIVE_SESSIONS[entity_id]
    sess['dissonance']  = dissonance
    sess['last_action'] = action
    sess['last_ts']     = ts
    if action not in sess['restrictions']:
        sess['restrictions'].append(action)
    if action == 'isolate_session':   sess['phase'] = 'sandbox'
    elif action == 'terminate_session': sess['phase'] = 'terminated'
    elif action in ('reduce_write_permissions', 'increase_validation_weight'):
        if sess['phase'] == 'normal':
            sess['phase'] = 'alert'

    return rec


print('✅ Motor de ejecución por nivel listo.')
print('   dry_run=True disponible para probar sin enviar nada al exterior.')

## Bloque 4 — Monitor de disonancia y respuesta adaptativa

Procesa eventos de disonancia del Modo 1 (o en tiempo real si hay stream).
Por cada evento que cruza un umbral, el PolicyAdapter selecciona la acción
y el dispatcher la enruta según nivel.

In [ ]:
# ─── Simulador del escenario del despacho ────────────────────
# Replica exactamente el escenario narrativo:
# Lunes 10:47 — Luis Colaborador Ext. desde IP desconocida

def simulate_attack_events(cfg: dict, seed: int = 42) -> List[dict]:
    """Genera la secuencia de eventos del ataque para el monitor."""
    import random
    rng   = random.Random(seed)
    base  = datetime(2025, 1, 20, 10, 47, 0, tzinfo=timezone.utc)
    users = {u['id']: u for u in cfg['users']}
    assets = {a['id']: a for a in cfg['assets']}

    attacker_id = 'u02'
    attack_ip   = '203.45.67.89'
    targeted    = ['a01', 'a02', 'a03']

    sequence = [
        (0,  attacker_id, attack_ip, 'login',  '',          'Inicio sesión IP desconocida'),
        (2,  attacker_id, attack_ip, 'read',   targeted[0], 'Lectura expediente principal'),
        (8,  attacker_id, attack_ip, 'read',   targeted[0], 'Lectura expediente (continua)'),
        (12, attacker_id, attack_ip, 'export', targeted[0], 'Intento de exportación 1'),
        (14, attacker_id, attack_ip, 'export', targeted[0], 'Reintento exportación'),
        (16, attacker_id, attack_ip, 'read',   targeted[1], 'Lectura 2do expediente'),
        (20, attacker_id, attack_ip, 'modify', targeted[1], 'Intento modificación fecha'),
        (24, attacker_id, attack_ip, 'read',   targeted[2], 'Lectura 3er activo'),
        (28, attacker_id, attack_ip, 'export', targeted[1], 'Intento exportación 2'),
        (33, attacker_id, attack_ip, 'read',   targeted[0], 'Relectura expediente'),
        # Tráfico normal de otros usuarios (entrelazado)
        (5,  'u01', '192.168.1.10', 'read',   'a04',       'Ana — plantillas'),
        (10, 'u03', '192.168.1.12', 'write',  'a05',       'María — agenda'),
        (18, 'u04', '192.168.1.13', 'read',   'a03',       'Carlos — contratos'),
        (25, 'u01', '192.168.1.10', 'modify', 'a01',       'Ana — expediente Alpha'),
    ]

    events = []
    for offset, uid, ip, action, asset_id, desc in sequence:
        ts   = base + __import__('datetime').timedelta(minutes=offset)
        user = users.get(uid, {})
        asset = assets.get(asset_id, {'sensitivity': 'low', 'name': 'Sistema'})
        # Calcular disonancia simulada
        is_att = (uid == attacker_id)
        dis = 0.0
        if ip not in user.get('typical_ips', []):       dis += 0.22
        if user.get('role') == 'external' and \
           action in ('export','modify'):                dis += 0.15
        if asset.get('sensitivity') == 'high' and \
           user.get('role') == 'external':              dis += 0.08
        # Acumulación
        prev = [e['dissonance'] for e in events if e['entity_id'] == uid]
        if prev:
            dis += np.mean(prev[-3:]) * 0.20
        dis += rng.uniform(0.02, 0.06) if is_att else rng.uniform(0.00, 0.02)
        dis  = float(np.clip(dis, 0.0, 1.0))

        events.append({
            'timestamp':   ts.isoformat(),
            'minutes':     offset,
            'entity_id':   uid,
            'source_ip':   ip,
            'action':      action,
            'asset_id':    asset_id,
            'asset_name':  asset.get('name', ''),
            'sensitivity': asset.get('sensitivity', 'low'),
            'dissonance':  round(dis, 4),
            'description': desc,
            'is_attacker': is_att,
        })

    events.sort(key=lambda e: e['minutes'])
    return events


def process_events(
    events:  List[dict],
    adapter: PolicyAdapter,
    cfg:     dict,
    dry_run: bool = True,
) -> List[dict]:
    """
    Procesa la secuencia de eventos:
    para cada uno que supera umbral → recomienda → despacha → registra.
    """
    th_alert   = cfg['thresholds']['geo_dissonance_alert']
    th_isolate = cfg['thresholds']['geo_dissonance_isolate']
    processed  = []

    for evt in events:
        dis  = evt['dissonance']
        eid  = evt['entity_id']
        recommendations = adapter.recommend(dis, eid, context=evt)
        dispatched = []

        if dis >= th_alert and recommendations:
            # Ejecutar top recomendaciones (máx 3 por evento)
            for act, score in recommendations[:3]:
                rec = dispatch_action(act, eid, dis, context=evt, dry_run=dry_run)
                dispatched.append({'action': act, 'score': score,
                                   'record_id': rec.record_id,
                                   'result': rec.execution_result,
                                   'override_token': rec.override_token})

        processed.append({
            **evt,
            'recommendations': recommendations,
            'dispatched':      dispatched,
            'phase': ACTIVE_SESSIONS.get(eid, {}).get('phase', 'normal'),
        })

    return processed


# ─── Ejecutar simulación ──────────────────────────────────────
print('⚙️  Simulando secuencia de ataque (dry_run=True)...')
print('   Para ejecución real: cambiar dry_run=False')
print()

EVENTS = simulate_attack_events(CFG)
PROCESSED = process_events(EVENTS, adapter, CFG, dry_run=True)

th_a = CFG['thresholds']['geo_dissonance_alert']
triggered = [e for e in PROCESSED if e['dissonance'] >= th_a]
print(f'✅ Eventos procesados  : {len(PROCESSED)}')
print(f'   Sobre umbral ({th_a}) : {len(triggered)}')
print(f'   Registros forenses  : {len(FORENSIC_LOG)}')
print()
print('Primeras 5 acciones despachadas:')
for rec in FORENSIC_LOG[:5]:
    print(f'  [{rec.record_id}] {rec.action:<35} eid={rec.entity_id} '
          f'dis={rec.dissonance:.3f} L{rec.level} → {rec.execution_result[:60]}')

## Bloque 5 — Override humano y señal de entrenamiento

El administrador puede revertir cualquier acción ejecutada usando el `override_token`.
El override actualiza el PolicyAdapter con `W_CORRECTION = 2.0` — el sistema aprende
que esa acción en ese contexto era incorrecta.

El correction_rate bimestral es el KPI principal del Modo 2:
debe bajar con cada ciclo.

In [ ]:
# ─── Sistema de override ──────────────────────────────────────

def apply_override(
    override_token: str,
    overriding_admin: str,
    correct_action:   str = None,   # None = falso positivo, ninguna acción era correcta
    low_clarity:      bool = False,
) -> dict:
    """
    Revierte una acción ejecutada y registra la corrección.
    Actualiza el PolicyAdapter con la señal de entrenamiento.
    """
    # Buscar el registro
    rec = next((r for r in FORENSIC_LOG
                if r.override_token == override_token), None)
    if not rec:
        return {'ok': False, 'msg': f'Token no encontrado: {override_token}'}
    if rec.override:
        return {'ok': False, 'msg': f'Ya fue sobreescrito: {rec.record_id}'}

    # Marcar override en el registro
    rec.override        = True
    rec.override_by     = overriding_admin
    rec.override_action = correct_action or 'false_positive'
    rec.override_ts     = datetime.now(timezone.utc).isoformat()

    # Actualizar PolicyAdapter
    adapter.record_decision(
        action=rec.action,
        dissonance=rec.dissonance,
        confirmed=False,
        overridden_to=correct_action,
        low_clarity=low_clarity,
    )

    # Revertir estado de sesión si es posible
    sess = ACTIVE_SESSIONS.get(rec.entity_id, {})
    if rec.action in sess.get('restrictions', []):
        sess['restrictions'].remove(rec.action)
    if not sess.get('restrictions'):
        sess['phase'] = 'normal'

    return {
        'ok':         True,
        'record_id':  rec.record_id,
        'action':     rec.action,
        'entity_id':  rec.entity_id,
        'overridden_to': correct_action or 'false_positive',
        'correction_rate': round(adapter.correction_rate, 4),
        'msg': (f'Override aplicado. PolicyAdapter actualizado. '
                f'Correction rate: {adapter.correction_rate:.2%}')
    }


def confirm_action(override_token: str, confirming_admin: str) -> dict:
    """Confirma que una acción fue correcta — refuerza el PolicyAdapter."""
    rec = next((r for r in FORENSIC_LOG
                if r.override_token == override_token), None)
    if not rec:
        return {'ok': False, 'msg': f'Token no encontrado: {override_token}'}

    adapter.record_decision(
        action=rec.action,
        dissonance=rec.dissonance,
        confirmed=True,
    )
    return {
        'ok': True,
        'record_id': rec.record_id,
        'action': rec.action,
        'correction_rate': round(adapter.correction_rate, 4),
        'msg': f'Acción confirmada. Peso reforzado en PolicyAdapter.'
    }


# ─── Reporte bimestral del adapter ───────────────────────────

def adapter_bimester_report(adapter: PolicyAdapter) -> str:
    hist = adapter.history
    if not hist:
        return '**Sin historial de decisiones en este bimestre.**'
    n_total    = len(hist)
    n_correct  = sum(1 for h in hist if h['confirmed'])
    n_override = sum(1 for h in hist if not h['confirmed'])
    cr         = adapter.correction_rate

    lines = [
        f'## 🧠 Reporte del PolicyAdapter — {adapter.bimester}',
        f'| Métrica | Valor |',
        f'|---|---|',
        f'| Decisiones totales | {n_total} |',
        f'| Confirmadas | {n_correct} |',
        f'| Overrides (correcciones) | {n_override} |',
        f'| **Correction rate** | **{cr:.2%}** |',
        f'',
        '### Pesos actuales por acción y contexto',
        '| Acción | low | med | high | critical |',
        '|---|---|---|---|---|',
    ]
    for act, buckets in adapter.weights.items():
        lines.append(
            f'| {act} | {buckets["low"]:.2f} | {buckets["med"]:.2f} '
            f'| {buckets["high"]:.2f} | {buckets["critical"]:.2f} |'
        )
    lines += [
        '',
        '### Interpretación del correction rate',
        '| Rango | Significado |',
        '|---|---|',
        '| > 40% | Adapter en período de aprendizaje inicial — normal en B1 |',
        '| 20–40% | Adapter convergiendo — revisar umbrales |',
        '| 10–20% | Adapter maduro — alta precisión |',
        '| < 10%  | Adapter estabilizado — considerar expandir acciones delegadas |',
    ]
    return '\n'.join(lines)


print('✅ Sistema de override listo.')
print(f'   Tokens disponibles para override: {len(FORENSIC_LOG)}')
if FORENSIC_LOG:
    sample = FORENSIC_LOG[0]
    print(f'   Ejemplo: token={sample.override_token} → {sample.action}')

## Bloque 6 — Dashboard integrado

In [ ]:
import matplotlib
matplotlib.use('Agg')
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr

PHASE_COLORS = {
    'normal':     '#378ADD',
    'alert':      '#BA7517',
    'sandbox':    '#D85A30',
    'terminated': '#E24B4A',
}


def plot_session_timeline(processed: List[dict], cfg: dict) -> go.Figure:
    th_a = cfg['thresholds']['geo_dissonance_alert']
    th_i = cfg['thresholds']['geo_dissonance_isolate']

    entities = list({e['entity_id'] for e in processed})
    palette  = ['#E24B4A','#378ADD','#639922','#BA7517','#9B59B6']
    fig = go.Figure()

    for i, eid in enumerate(entities):
        evts = sorted([e for e in processed if e['entity_id'] == eid],
                      key=lambda x: x['minutes'])
        x = [e['minutes'] for e in evts]
        y = [e['dissonance'] for e in evts]
        texts = [f"{e['description']}<br>{e['action']} → {e['asset_name']}<br>fase: {e['phase']}" for e in evts]
        col = palette[i % len(palette)]
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines+markers', name=eid,
            line=dict(color=col, width=2),
            marker=dict(size=8, color=col),
            text=texts,
            hovertemplate='<b>%{x} min</b><br>Dis: %{y:.3f}<br>%{text}<extra></extra>',
        ))

    fig.add_hline(y=th_a, line_dash='dash', line_color='#BA7517', line_width=1,
                  annotation_text=f'alerta {th_a}', annotation_font_size=9)
    fig.add_hline(y=th_i, line_dash='dash', line_color='#D85A30', line_width=1,
                  annotation_text=f'sandbox {th_i}', annotation_font_size=9)

    fig.update_layout(
        title='Disonancia por sesión — tiempo real',
        xaxis_title='Minutos', yaxis_title='Disonancia',
        yaxis_range=[0, 1.05], height=350,
        margin=dict(l=40, r=120, t=60, b=40),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=11),
        yaxis_gridcolor='#e8e8e4',
        legend=dict(orientation='v', x=1.02),
    )
    return fig


def plot_adapter_weights(adapter: PolicyAdapter) -> go.Figure:
    actions = list(adapter.weights.keys())
    buckets = ['low', 'med', 'high', 'critical']
    colors  = ['#639922','#BA7517','#D85A30','#E24B4A']
    fig = go.Figure()
    for j, bucket in enumerate(buckets):
        vals = [adapter.weights[a][bucket] for a in actions]
        fig.add_trace(go.Bar(
            name=f'ctx: {bucket}', x=actions, y=vals,
            marker_color=colors[j], opacity=0.85,
        ))
    fig.update_layout(
        barmode='group', title='Pesos del PolicyAdapter por acción y contexto',
        height=320, margin=dict(l=40, r=40, t=60, b=80),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=10),
        yaxis_gridcolor='#e8e8e4', yaxis_title='Peso',
        xaxis_tickangle=-30,
        legend=dict(orientation='h', y=-0.25),
    )
    return fig


# ─── Dashboard ────────────────────────────────────────────────

def run_full_simulation(config_json: str, dry_run_chk: bool, seed: int):
    global FORENSIC_LOG, ACTIVE_SESSIONS, PROCESSED, adapter
    FORENSIC_LOG    = []
    ACTIVE_SESSIONS = {}

    try:
        cfg = json.loads(config_json) if config_json.strip() else CFG
        merged = {**DEFAULT_CONFIG, **cfg}
        for k in ('thresholds','actions_dict','action_levels'):
            if k in DEFAULT_CONFIG and k in cfg:
                merged[k] = {**DEFAULT_CONFIG[k], **cfg[k]}
    except Exception as e:
        return None, None, None, f'❌ Config inválida: {e}', '', '', '', '', []

    global ACTION_LEVELS
    ACTION_LEVELS = merged.get('action_levels', {k:0 for k in ACTIONS_ORDERED})

    evts = simulate_attack_events(merged, seed=int(seed))
    proc = process_events(evts, adapter, merged, dry_run=dry_run_chk)
    PROCESSED = proc

    fig_ses = plot_session_timeline(proc, merged)
    fig_adp = plot_adapter_weights(adapter)

    from IPython.display import Markdown
    report_md = adapter_bimester_report(adapter)

    # Métricas
    th_a  = merged['thresholds']['geo_dissonance_alert']
    m_rec = str(len(FORENSIC_LOG))
    m_exe = str(sum(1 for r in FORENSIC_LOG if r.executed))
    m_ove = str(sum(1 for r in FORENSIC_LOG if r.override))
    m_cr  = f'{adapter.correction_rate:.2%}'
    m_phase = ', '.join(f'{eid}:{s["phase"]}'
                        for eid, s in ACTIVE_SESSIONS.items()) or 'sin sesiones'

    # Tabla de registros forenses
    table = []
    for rec in FORENSIC_LOG:
        ov = '↩️' if rec.override else ('✅' if rec.executed else '⏳')
        table.append([
            rec.record_id, rec.entity_id, rec.action,
            f'L{rec.level}', f'{rec.dissonance:.3f}',
            ov, rec.override_token, rec.sha256
        ])

    return (fig_ses, fig_adp, report_md,
            m_rec, m_exe, m_ove, m_cr, m_phase, table)


def do_override(token: str, admin: str, correct_act: str):
    result = apply_override(
        override_token=token.strip(),
        overriding_admin=admin.strip(),
        correct_action=correct_act.strip() or None,
    )
    fig_adp = plot_adapter_weights(adapter)
    return result['msg'], fig_adp, f'{adapter.correction_rate:.2%}'


def do_confirm(token: str, admin: str):
    result = confirm_action(token.strip(), admin.strip())
    fig_adp = plot_adapter_weights(adapter)
    return result['msg'], fig_adp, f'{adapter.correction_rate:.2%}'


DEFAULT_JSON_UI = json.dumps({
    'org_name':    CFG['org_name'],
    'bimester':    BIMESTER,
    'thresholds':  CFG['thresholds'],
    'action_levels': CFG.get('action_levels', {k:0 for k in ACTIONS_ORDERED}),
    'level1_webhook_url': '',
    'level2_api_base':    '',
}, indent=2)


with gr.Blocks(
    title='GSL Modo 2 — Respuesta Adaptativa',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate'),
) as demo:

    gr.Markdown(f"""
# 🛡️ GSL Modo 2 — Respuesta Adaptativa Integrada
**{CFG['org_name']}** · Bimestre {BIMESTER}

El PolicyAdapter selecciona y despacha acciones según nivel de permisos configurado.
El override humano actualiza los pesos — el sistema aprende bimestre a bimestre.
""")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown('### ⚙️ Configuración y niveles de despliegue')
            config_box = gr.Code(value=DEFAULT_JSON_UI, language='json',
                                 label='org_config (action_levels: 0=recom · 1=webhook · 2=api · 3=agente)',
                                 lines=18)
            with gr.Row():
                dry_run_chk = gr.Checkbox(value=True,
                    label='dry_run — simular sin enviar al exterior')
                seed_sl = gr.Slider(1, 999, value=42, step=1, label='Semilla')
            run_btn = gr.Button('▶ Ejecutar simulación', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('### 📊 Estado del sistema')
            with gr.Row():
                m_rec   = gr.Textbox(label='Registros forenses', interactive=False)
                m_exe   = gr.Textbox(label='Acciones ejecutadas', interactive=False)
            with gr.Row():
                m_ove   = gr.Textbox(label='Overrides aplicados', interactive=False)
                m_cr    = gr.Textbox(label='Correction rate',    interactive=False)
            m_phase = gr.Textbox(label='Fases de sesiones activas', interactive=False)

            gr.Markdown('### ↩️ Override / Confirmación')
            ov_token = gr.Textbox(placeholder='override_token del registro',
                                   label='Token')
            ov_admin = gr.Textbox(placeholder='ID del administrador',
                                   label='Admin')
            ov_act   = gr.Dropdown(
                choices=[''] + ACTIONS_ORDERED,
                value='', label='Acción correcta (vacío = falso positivo)')
            with gr.Row():
                ov_btn  = gr.Button('↩️ Override', variant='stop')
                cf_btn  = gr.Button('✅ Confirmar', variant='secondary')
            ov_result = gr.Textbox(label='Resultado', interactive=False)

    gr.Markdown('### 📈 Disonancia por sesión')
    fig_ses_out = gr.Plot()

    gr.Markdown('### 🧠 Pesos del PolicyAdapter')
    fig_adp_out = gr.Plot()

    gr.Markdown('### 📄 Reporte del adapter')
    report_out = gr.Markdown()

    gr.Markdown('### 🗂️ Registro forense inmutable')
    table_out = gr.Dataframe(
        headers=['ID','Entidad','Acción','Nivel',
                 'Disonancia','Estado','Token','SHA-256'],
        wrap=True)

    with gr.Accordion('📚 Guía de niveles y permisos', open=False):
        gr.Markdown("""
## Configuración de niveles en `org_config.json`

```json
{
  "action_levels": {
    "reduce_write_permissions":   1,  ← webhook
    "increase_validation_weight": 1,  ← webhook
    "snapshot_state":             2,  ← API directa
    "isolate_session":            0,  ← recomendación (un click)
    "terminate_session":          0,  ← recomendación (requiere confirmación)
    "revoke_credentials":         0,
    "freeze_affected_assets":     2,
    "generate_incident_report":   2
  },
  "level1_webhook_url": "https://tu-servidor.com/gsl-webhook",
  "level2_api_base":    "https://api.tu-sistema.com",
  "level2_api_key":     "sk-...",
  "level3_agent_socket": "/var/run/gsl-agent.sock"
}
```

## Payload Nivel 1 (webhook)
```json
{"event": "gsl_action", "action": "reduce_write_permissions",
 "entity_id": "u02", "dissonance": 0.31, "reversible": true,
 "override_token": "abc123", "ttl_minutes": 30}
```

## Evolución del correction rate bimestral
| Bimestre | Correction rate esperado |
|---|---|
| B1 | 40–50% — adapter aprendiendo |
| B2 | 25–35% — convergiendo |
| B3 | 12–20% — maduro |
| B4+ | < 10% — estabilizado |
""")

    run_btn.click(
        fn=run_full_simulation,
        inputs=[config_box, dry_run_chk, seed_sl],
        outputs=[fig_ses_out, fig_adp_out, report_out,
                 m_rec, m_exe, m_ove, m_cr, m_phase, table_out]
    )
    ov_btn.click(
        fn=do_override,
        inputs=[ov_token, ov_admin, ov_act],
        outputs=[ov_result, fig_adp_out, m_cr]
    )
    cf_btn.click(
        fn=do_confirm,
        inputs=[ov_token, ov_admin],
        outputs=[ov_result, fig_adp_out, m_cr]
    )

print('✅ Dashboard Modo 2 listo. Lanzando...')
demo.launch(share=False, inbrowser=True)

## Persistencia y ExportPackage

Guarda el PolicyAdapter actualizado para que el próximo bimestre
empiece desde el estado aprendido, no desde cero.

In [ ]:
import zipfile

# Guardar adapter
adapter.save(ARTIFACTS / 'policy_adapter.json')
# Copia persistente para el próximo bimestre
adapter.save(ADAPTER_PATH)

# Registro forense completo
forensic_records = [r.to_dict() for r in FORENSIC_LOG]
with open(ARTIFACTS / 'forensic_log.json', 'w') as f:
    json.dump(forensic_records, f, indent=2)

# Estado de sesiones
with open(ARTIFACTS / 'active_sessions.json', 'w') as f:
    json.dump(ACTIVE_SESSIONS, f, indent=2)

# Manifiesto
manifest = {
    'run_id':           RUN_ID,
    'mode':             'GSL-Modo2-Activo',
    'bimester':         BIMESTER,
    'org_name':         CFG['org_name'],
    'principal_id':     CFG['principal_id'],
    'generated_at':     datetime.now(timezone.utc).isoformat(),
    'forensic_records': len(FORENSIC_LOG),
    'actions_executed': sum(1 for r in FORENSIC_LOG if r.executed),
    'overrides':        sum(1 for r in FORENSIC_LOG if r.override),
    'correction_rate':  round(adapter.correction_rate, 4),
    'adapter_history':  len(adapter.history),
    'active_sessions':  {eid: s['phase'] for eid, s in ACTIVE_SESSIONS.items()},
}
with open(ARTIFACTS / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ExportPackage
export_zip = ARTIFACTS.parent.parent / \
    f'gsl_m2_{CFG["principal_id"]}_{BIMESTER}.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ARTIFACTS.iterdir():
        zf.write(f, arcname=f'modo2/{f.name}')

print(f'\n✅ PolicyAdapter guardado: {ADAPTER_PATH}')
print(f'   Correction rate final  : {adapter.correction_rate:.2%}')
print(f'   Decisiones en historial: {len(adapter.history)}')
print(f'\n✅ ExportPackage: {export_zip}')
print(f'   Tamaño: {export_zip.stat().st_size / 1024:.1f} KB')
print(f'\n📋 Manifiesto final:')
print(json.dumps({k: v for k, v in manifest.items()
                  if k != 'active_sessions'}, indent=2))